# Project Vyapaar | Notebook 03: Delivery & Customer Reviews 🚚

**Team:** ____________________

**Lead for this part:** Member 3 (Operations Analyst)

Is the customer experience about more than the product? Order → Delivery → Review.

### How to run this notebook
**On Google Colab:**
1. Upload this notebook to Colab (File → Upload notebook).
2. Click **Runtime → Run all**.
3. When the "Choose Files" button appears, select the 4 data files: `orders.csv`, `products.csv`, `reviews.csv`, `marketing.csv`.

**On Jupyter (laptop):** keep the 4 csv files in the same folder as this notebook (or in a `data` folder next to it) and run all cells.

## Step 0: Load the data

In [ ]:
import pandas as pd
import plotly.express as px
import os

# show numbers with 2 decimals and commas
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)

In [ ]:
# This small function finds a csv file whether it is in this folder or in a "data" folder
def find_file(name):
    for folder in ["", "data/", "../data/"]:
        if os.path.exists(folder + name):
            return folder + name
    return None

# If the files are not there we are probably on Google Colab, so ask to upload them
if find_file("orders.csv") is None:
    from google.colab import files
    print("Please upload orders.csv, products.csv, reviews.csv and marketing.csv")
    files.upload()

In [ ]:
# encoding="utf-8-sig" removes a hidden character at the start of orders.csv and reviews.csv
orders = pd.read_csv(find_file("orders.csv"), encoding="utf-8-sig")
products = pd.read_csv(find_file("products.csv"))
marketing = pd.read_csv(find_file("marketing.csv"))

# keep_default_na=False stops pandas from turning the word "None" into a blank.
# In reviews, "None" means "no complaint", so we want to keep it.
reviews = pd.read_csv(find_file("reviews.csv"), encoding="utf-8-sig", keep_default_na=False, na_values=[""])

print("orders:", orders.shape)
print("products:", products.shape)
print("reviews:", reviews.shape)
print("marketing:", marketing.shape)

## Step 1: Clean the data
These are the same cleaning steps as **Notebook 01 (Data Cleaning)**. The reasons for each step are explained there.

In [ ]:
# 1. remove exact duplicate rows
orders = orders.drop_duplicates()
products = products.drop_duplicates()
reviews = reviews.drop_duplicates()

# 2. fix the 2 wrongly labelled products
products.loc[products["product_id"] == "P011", "category"] = "Beauty"
products.loc[products["product_id"] == "P074", "subcategory"] = "Small Appliances"

# 3. fix spelling of sales channel
orders["sales_channel"] = orders["sales_channel"].replace("web site", "Website")

# 4. fill the 1 missing discount using: discount = 1 - revenue / (quantity x price)
orders["discount_pct"] = orders["discount_pct"].fillna(
    (1 - orders["revenue_inr"] / (orders["quantity"] * orders["unit_price_inr"])).round(2))

# 5. label blanks instead of deleting them
orders["customer_id"] = orders["customer_id"].fillna("Unknown")
orders["campaign_id"] = orders["campaign_id"].fillna("No campaign")

# 6. text -> dates
orders["order_date"] = pd.to_datetime(orders["order_date"])
marketing["start_date"] = pd.to_datetime(marketing["start_date"])
marketing["end_date"] = pd.to_datetime(marketing["end_date"])

# 7. join everything into one table
df = orders.merge(products, on="product_id").merge(reviews, on="order_id")
df["margin_%"] = df["profit_inr"] / df["revenue_inr"] * 100
print("Clean table:", df.shape)

---
# Stage 3: Customer & Operations Mystery 🚚
*We did not get `delivery.csv`, so we use the review complaint **"Late delivery"** as our sign of a delivery problem.*

### 3.1 Which complaint hurts the most?

In [ ]:
comp = df.groupby("complaint_category").agg({"order_id": "count", "rating": "mean", "escalated": "mean"})
comp.columns = ["orders", "avg_rating", "escalation_rate"]
comp["escalation_rate"] = comp["escalation_rate"] * 100
comp.sort_values("orders", ascending=False)

- **Late delivery** is the most common complaint (695 orders).
- It is also the worst one: average rating **2.43** and **48%** of these customers escalate. Product complaints are only escalated 7-14% of the time.

### 3.2 Where is it happening?

In [ ]:
df["late_delivery"] = df["complaint_category"] == "Late delivery"

region = df.groupby("region_at_order").agg({"order_id": "count", "late_delivery": "mean", "escalated": ["mean", "sum"], "rating": "mean"})
region.columns = ["orders", "late_delivery_%", "escalation_%", "escalations", "avg_rating"]
region["late_delivery_%"] = region["late_delivery_%"] * 100
region["escalation_%"] = region["escalation_%"] * 100
region["share_of_escalations_%"] = region["escalations"] / region["escalations"].sum() * 100
region

In [ ]:
# CHART 3
chart3 = region.reset_index()
fig = px.bar(chart3, x="region_at_order", y=["late_delivery_%", "escalation_%"], barmode="group", text_auto=".1f",
             title="Chart 3: East has 3x the late deliveries and about 4x the escalations of other regions",
             labels={"region_at_order": "Region", "value": "% of orders", "variable": ""})
fig.show()

In [ ]:
# Is it just one bad month? (% late-delivery complaints by month)
df["month"] = df["order_date"].dt.month
(df.pivot_table(index="month", columns="region_at_order", values="late_delivery", aggfunc="mean") * 100).round(0)

In [ ]:
# Is it one sales channel?
(df.pivot_table(index="region_at_order", columns="sales_channel", values="late_delivery", aggfunc="mean") * 100).round(0)

**What we found:**
- **38.5%** of East orders get a late-delivery complaint. In the other regions it is only **12-13%**.
- East is **19% of orders** but **48% of all escalations**, and its rating is **3.23** vs about 3.9 elsewhere.
- It is high in **every month** and **every channel**, so it is a permanent East delivery problem, not a one-time thing.

### 3.3 Do unhappy customers stop buying?

In [ ]:
known = df[df["customer_id"] != "Unknown"]
per_region = known.groupby("region_at_order").agg({"order_id": "count", "customer_id": "nunique"})
per_region["orders_per_customer"] = per_region["order_id"] / per_region["customer_id"]
per_region

In [ ]:
# Small thing we noticed: Store orders also get late-delivery complaints
store = df[df["sales_channel"] == "Store"]
print("Store orders with a late-delivery complaint:", store["late_delivery"].sum(), "out of", len(store))

**Honest result:** East customers still buy about **6.5 times** a year, which is close to the other regions (6.6-6.9). So with one year of data we **cannot prove** late delivery makes customers leave.

**Why it still matters:**
1. Every escalation costs customer-service time and money, and East creates almost half of them.
2. A 3.2 rating is visible to new customers on apps and marketplaces.
3. Loyal customers may stay only until a competitor delivers faster.

*Also:* 179 **Store** orders have late-delivery complaints, which is strange for an in-store purchase. We need `delivery.csv` to check this.

**Recommendation:** audit the East delivery partner and promised delivery dates, and get `delivery.csv` to find the exact cause.